# CEG-WM Stage-A HF-v2 rank-gate confirmation

Run all cells after setting Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN`. The notebook resolves the HF-v2 rank-gate branch head once, checks it out detached, and automatically reuses a verified final package or the highest compatible checkpoint. It runs only the frozen untouched identity confirmation roster. LPIPS and attacks remain unmeasured, and the returned package awaits independent Agent5 adjudication.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os
def _required_secret(name):
    value = os.environ.pop(name, None)
    if value is None:
        value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f'missing required Colab Secret: {name}')
    return value
root_key = _required_secret('CEG_WM_ROOT_KEY')
hf_token = _required_secret('HF_TOKEN')
run_store_root = Path('/content/drive/MyDrive/CEG-WM/stage_a_hf_v2_rankgate')
run_store_root.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, re, subprocess, sys
repo = Path('/content/CEG-WM-stage-a-hf-v2-rankgate-exact')
if repo.exists():
    raise RuntimeError('detached checkout path already exists')
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/RICHAAARC/CEG-WM.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', 'refs/heads/stage-a-hf-v2-rankgate'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
resolved_exact = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if re.fullmatch(r'[0-9a-f]{40}', resolved_exact) is None:
    raise RuntimeError('resolved Stage-A branch head is not an exact revision')
if subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout:
    raise RuntimeError('execution checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)


In [ ]:
local_output_root = Path('/content/cegwm-stage-a-hf-v2-rankgate-local')
runner_env = dict(os.environ)
runner_env['CEG_WM_ROOT_KEY'] = root_key
runner_env['HF_TOKEN'] = hf_token
command = [sys.executable, '-m', 'experiments.stage_a.run_hf_a2_colab', '--repo-root', str(repo), '--output-root', str(local_output_root), '--expected-exact', resolved_exact, '--run-store-root', str(run_store_root)]
run_id = None
fatal_event = None
try:
    process = subprocess.Popen(command, cwd=str(repo), env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    for line in process.stdout:
        if line.startswith('CEGWM_PROGRESS '):
            progress = json.loads(line.removeprefix('CEGWM_PROGRESS '))
            if not set(progress).issubset({'run_id', 'committed', 'fixed_total', 'phase'}):
                raise RuntimeError('runner progress exposed unexpected fields')
            candidate_run_id = progress.get('run_id')
            if re.fullmatch(r'a2hfv2-[0-9a-f]{24}', candidate_run_id or '') is None:
                raise RuntimeError('runner progress has invalid deterministic run identity')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
            print({'run_id': run_id, 'committed': progress['committed'], 'fixed_total': progress['fixed_total']})
        elif line.startswith('CEGWM_FATAL '):
            fatal_event = json.loads(line.removeprefix('CEGWM_FATAL '))
            if set(fatal_event) != {'run_id', 'error_class', 'export_status'}:
                raise RuntimeError('runner fatal event exposed unexpected fields')
            candidate_run_id = fatal_event.get('run_id')
            if re.fullmatch(r'a2hfv2-[0-9a-f]{24}', candidate_run_id or '') is None:
                raise RuntimeError('runner fatal event has invalid deterministic run identity')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
    runner_rc = process.wait()
finally:
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    root_key = hf_token = ''
    del root_key, hf_token, runner_env
if run_id is None:
    raise RuntimeError('runner produced no deterministic run identity')


In [ ]:
import hashlib, zipfile
drive_run_dir = run_store_root / run_id
if runner_rc == 2:
    if fatal_event is None or fatal_event['export_status'] != 'published':
        raise RuntimeError('runner RC2 has no published sanitized failure package')
    error_class = fatal_event['error_class']
    if error_class not in {'initialization_failure', 'resume_validation_failure', 'runtime_execution_failure', 'checkpoint_failure', 'final_export_failure'}:
        raise RuntimeError('runner RC2 error class is not predeclared')
    zip_path = drive_run_dir / f'failure-{error_class}.zip'
else:
    if fatal_event is not None or runner_rc not in {0, 1}:
        raise RuntimeError('runner terminal event/RC mismatch')
    zip_path = drive_run_dir / f'{run_id}.zip'
checksum_path = drive_run_dir / f'{zip_path.name}.sha256'
if not (zip_path.is_file() and checksum_path.is_file()):
    raise RuntimeError('runner did not publish a complete terminal package pair')
checksum_parts = checksum_path.read_text().strip().split()
if len(checksum_parts) != 2 or checksum_parts[1] != zip_path.name:
    raise RuntimeError('terminal checksum file is malformed')
zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
if checksum_parts[0] != zip_sha256:
    raise RuntimeError('terminal ZIP checksum mismatch')
with zipfile.ZipFile(zip_path) as archive:
    if set(archive.namelist()) != {'receipt.json', 'result.json'}:
        raise RuntimeError('terminal ZIP members mismatch')
    receipt = json.loads(archive.read('receipt.json'))
    result = json.loads(archive.read('result.json'))
if receipt['run_id'] != run_id or result['run_id'] != run_id:
    raise RuntimeError('terminal run identity mismatch')
if receipt['resolved_exact'] != resolved_exact or result['resolved_exact'] != resolved_exact:
    raise RuntimeError('terminal exact mismatch')
if receipt['rc'] != runner_rc or result['rc'] != runner_rc:
    raise RuntimeError('runner/terminal RC mismatch')
if result['fixed_unit_count'] != 8 or result['fixed_record_count'] != 16:
    raise RuntimeError('terminal fixed denominator identity mismatch')
if runner_rc == 2:
    if receipt['error_class'] != error_class or result['error_class'] != error_class or receipt['status'] != 'operational_failure':
        raise RuntimeError('RC2 failure identity mismatch')
    committed = result['committed_unit_ids']
    roster = result['ordered_roster_unit_ids']
    records = result['records']
    if committed != roster[:len(committed)] or result['committed_unit_count'] != len(committed) or len(records) != len(committed) * 2:
        raise RuntimeError('RC2 committed prefix mismatch')
    for index, unit_id in enumerate(committed):
        pair = records[index * 2:index * 2 + 2]
        if [record['unit_id'] for record in pair] != [unit_id, unit_id] or [record['arm'] for record in pair] != ['hf_anchor', 'primary_null']:
            raise RuntimeError('RC2 paired record prefix mismatch')
elif len(result['records']) != 16:
    raise RuntimeError('final fixed denominator mismatch')
summary = {'run_id': run_id, 'resolved_exact': resolved_exact, 'status': receipt['status'], 'rc': runner_rc, 'drive_relative_dir': f'CEG-WM/stage_a_hf_v2_rankgate/{run_id}', 'zip_sha256': zip_sha256}
print(summary)
if runner_rc != 0:
    raise RuntimeError('runner completed with retained operational failures')
